<a href="https://colab.research.google.com/github/Rupesh756/Brain-tumor-Detection-using-deep-learning/blob/main/brain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
train_dir = "/content/drive/MyDrive/Colab Notebooks/brain/unzipdataset/Training"
test_dir  = "/content/drive/MyDrive/Colab Notebooks/brain/unzipdataset/Testing"

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_data = train_datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

test_data = test_datagen.flow_from_directory(
    test_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

Found 4492 images belonging to 4 classes.
Found 1599 images belonging to 4 classes.


In [ ]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, models

IMG_SIZE = 224

base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

# Freeze most layers
for layer in base_model.layers[:-20]:
    layer.trainable = False

# Custom head
x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.5)(x)
output = layers.Dense(4, activation='softmax')(x)

model = models.Model(inputs=base_model.input, outputs=output)

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    train_data,
    validation_data=test_data,
    epochs=15
)

Epoch 1/15
141/141 ━━━━━━━━━━━━━━━━━━━━ 1870s 13s/step - accuracy: 0.3724 - loss: 1.6253 - val_accuracy: 0.2495 - val_loss: 1.5053
Epoch 2/15
141/141 ━━━━━━━━━━━━━━━━━━━━ 596s 4s/step - accuracy: 0.4069 - loss: 1.4359 - val_accuracy: 0.2495 - val_loss: 1.6760
Epoch 3/15
141/141 ━━━━━━━━━━━━━━━━━━━━ 608s 4s/step - accuracy: 0.4323 - loss: 1.3287 - val_accuracy: 0.2514 - val_loss: 1.5405
Epoch 4/15
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.4519 - loss: 1.3226

### Image Validation Script

This script will go through your training and testing directories, attempt to open each file, and report any that cause an `UnidentifiedImageError`.

In [ ]:
from PIL import Image
import os

def validate_images_in_directory(directory_path):
    invalid_images = []
    print(f"\nValidating images in: {directory_path}")
    for root, _, files in os.walk(directory_path):
        for file in files:
            file_path = os.path.join(root, file)
            try:
                # Attempt to open the image
                with Image.open(file_path) as img:
                    img.verify() # Verify that it is an image
                # print(f"Successfully validated: {file_path}") # Uncomment for verbose output
            except (IOError, SyntaxError) as e:
                # Catch PIL errors (e.g., UnidentifiedImageError)
                invalid_images.append((file_path, str(e)))
                print(f"[ERROR] Invalid image detected: {file_path} - {e}")
            except Exception as e:
                # Catch any other unexpected errors
                invalid_images.append((file_path, str(e)))
                print(f"[ERROR] Unexpected error with file: {file_path} - {e}")

    if not invalid_images:
        print("No invalid images found in this directory.")
    else:
        print(f"Found {len(invalid_images)} invalid images in {directory_path}:")
        for img_path, error_msg in invalid_images:
            print(f"- {img_path}: {error_msg}")
    return invalid_images

# Validate training data directory
print("Starting validation for training directory...")
invalid_train_images = validate_images_in_directory(train_dir)

# Validate testing data directory
print("\nStarting validation for testing directory...")
invalid_test_images = validate_images_in_directory(test_dir)

print("\nImage validation complete.")

if invalid_train_images or invalid_test_images:
    print("\nPlease remove or replace the listed invalid files from your dataset directories.")
else:
    print("All images validated successfully. You can now retry training your model.")


It appears the file `/content/drive/MyDrive/Colab Notebooks/brain/unzipdataset/Testing/notumor/Te-no_167.jpg` is corrupted or not a valid image. You can remove it using the following code.

In [ ]:
import os

file_to_remove = '/content/drive/MyDrive/Colab Notebooks/brain/unzipdataset/Testing/notumor/Te-no_167.jpg'

if os.path.exists(file_to_remove):
    os.remove(file_to_remove)
    print(f"Removed: {file_to_remove}")
else:
    print(f"File not found: {file_to_remove}")


After removing the invalid file, please re-run the cell `jOc35Wu53zqV` to re-initialize your `ImageDataGenerator` objects (`train_data` and `test_data`) with the corrected dataset. Then you can retry training your model with cell `1TnlKiJw4ct6`.

In [ ]:
plt.plot(history.history['accuracy'], label='train_acc')
plt.plot(history.history['val_accuracy'], label='val_acc')
plt.legend()
plt.title("Accuracy")
plt.show()

plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.legend()
plt.title("Loss")
plt.show()

In [ ]:
loss, acc = model.evaluate(test_data)
print("Test Accuracy:", acc)

In [ ]:
model.save("/content/drive/MyDrive/brain_tumor_model.keras")

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing import image

img_path = "/content/drive/MyDrive/Colab Notebooks/brain/unzipdataset/Testing/glioma/Te-gl_1.jpg"  # change path
img = image.load_img(img_path, target_size=(224,224))
img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

prediction = model.predict(img_array)
class_names = list(train_data.class_indices.keys())

print("Predicted:", class_names[np.argmax(prediction)])

In [ ]:
from tensorflow.keras.models import load_model

model = load_model("/content/drive/MyDrive/brain_tumor_model.keras")

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = 224
BATCH_SIZE = 32

test_dir = "/content/drive/MyDrive/Colab Notebooks/brain/unzipdataset/Testing"

test_datagen = ImageDataGenerator(rescale=1./255)

test_data = test_datagen.flow_from_directory(
    test_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

In [ ]:
loss, acc = model.evaluate(test_data)
print("Test Accuracy:", acc)

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing import image

img_path = "/content/drive/MyDrive/Colab Notebooks/brain/unzipdataset/Testing/glioma/Te-gl_102.jpg"  # change this

img = image.load_img(img_path, target_size=(224,224))
img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

prediction = model.predict(img_array)

class_names = list(test_data.class_indices.keys())
print("Predicted:", class_names[np.argmax(prediction)])

In [ ]:
!pip install scikit-learn

In [ ]:
import numpy as np

# Predict on test data
y_pred = model.predict(test_data)
y_pred_classes = np.argmax(y_pred, axis=1)

y_true = test_data.classes
class_labels = list(test_data.class_indices.keys())

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_true, y_pred_classes, target_names=class_labels))

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y_true, y_pred_classes)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d',
            xticklabels=class_labels,
            yticklabels=class_labels)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
plt.plot(history.history['accuracy'], label='train_acc')
plt.plot(history.history['val_accuracy'], label='val_acc')
plt.legend()
plt.title("Accuracy")
plt.show()

plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.legend()
plt.title("Loss")
plt.show()